# Ejercicio 2: regularizar una regresión que se sobreajusta

El mismo `viajes_features.csv` y la misma preparación importada de `preparacion_taxis.py`. Mismo esqueleto del Ejercicio 1; `evaluar` acepta además `grado_interacciones=3`, que agrega como columnas todos los productos entre columnas hasta grado 3.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo tienen la comprobación. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**).

In [1]:
import time

import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from preparacion_taxis import dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (2380, 10) Prueba: (595, 10)


## La función que evalúa cada modelo

In [2]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo, grado_interacciones=None):
    """Igual que en el Ejercicio 1. Con grado_interacciones, agrega despues
    de la preparacion un PolynomialFeatures de ese grado: todos los
    productos entre columnas hasta ese grado, como columnas nuevas."""
    pasos = [("preparar", preparacion())]
    if grado_interacciones is not None:
        pasos.append(("interacciones", PolynomialFeatures(degree=grado_interacciones, include_bias=False)))
    pasos.append(("modelo", modelo))
    pipeline = Pipeline(pasos)
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {
        "modelo": nombre,
        "mae_entrenamiento": round(mae_ent, 4),
        "mae_prueba": round(mae_prueba, 4),
        "brecha": round(mae_prueba - mae_ent, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:28s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:.4f}")

## Paso 1: línea base

In [3]:
# TODO 1: la misma linea base del Ejercicio 1, con el nombre "linea base".
evaluar("linea base", DummyRegressor(strategy="median"))

linea base                   entrenamiento 13.1998   prueba 12.5308


## Paso 2: regresión lineal con interacciones de grado 3

In [4]:
# TODO 2: una regresion lineal sobre las interacciones de grado 3, con el
# nombre "lineal grado 3". Guarda en columnas_grado_3 cuantas columnas
# recibe el modelo (el atributo n_output_features_ del paso
# "interacciones" del pipeline entrenado, que evaluar guarda en
# entrenados["lineal grado 3"]).
evaluar("lineal grado 3", LinearRegression(), grado_interacciones=3)
columnas_grado_3 = entrenados["lineal grado 3"].named_steps["interacciones"].n_output_features_
print("Columnas con interacciones de grado 3:", columnas_grado_3, "| filas de entrenamiento:", len(X_ent))

lineal grado 3               entrenamiento  0.8922   prueba 1695287900.1597
Columnas con interacciones de grado 3: 3275 | filas de entrenamiento: 2380


## Paso 3: la misma regresión, con Ridge

In [5]:
# TODO 3: la misma regresion, regularizada con Ridge(alpha=10.0), con el
# nombre "ridge grado 3".
evaluar("ridge grado 3", Ridge(alpha=10.0), grado_interacciones=3)

ridge grado 3                entrenamiento  1.0272   prueba 1.3209


## Paso 4: un ensamblaje por boosting

In [6]:
# Un ensamblaje por boosting, con el nombre "gradient boosting".
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))

gradient boosting            entrenamiento  1.0705   prueba 1.2754


## Paso 5: la tabla y el mejor modelo

In [7]:
# Guarda la tabla en resultados_ejercicio2.csv y deja en "mejor" el
# nombre del modelo con menor error en prueba.
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_ejercicio2.csv", index=False)
mejor = tabla.loc[tabla["mae_prueba"].idxmin(), "modelo"]
print(tabla.to_string(index=False))
print("Mejor modelo:", mejor)

           modelo  mae_entrenamiento   mae_prueba        brecha  segundos
       linea base            13.1998 1.253080e+01 -6.690000e-01     0.016
   lineal grado 3             0.8922 1.695288e+09  1.695288e+09     4.989
    ridge grado 3             1.0272 1.320900e+00  2.938000e-01     0.956
gradient boosting             1.0705 1.275400e+00  2.048000e-01     0.277
Mejor modelo: gradient boosting


## Comprobación

In [8]:
import os


def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


def fila(nombre):
    return resultados.get(nombre)


comprobar(1, fila("linea base") is not None and 12.0 < fila("linea base")["mae_prueba"] < 13.0)
comprobar(2, fila("lineal grado 3") is not None and fila("lineal grado 3")["brecha"] > 100 and columnas_grado_3 == 3275)
comprobar(3, fila("ridge grado 3") is not None and fila("ridge grado 3")["mae_prueba"] < 1.5)
comprobar(4, fila("gradient boosting") is not None and fila("gradient boosting")["mae_prueba"] < 1.35)
comprobar(5, mejor == "gradient boosting" and os.path.exists("resultados_ejercicio2.csv"))

Paso 1: OK
Paso 2: OK
Paso 3: OK
Paso 4: OK
Paso 5: OK
